# Question 18 - 12. Integer to Roman

Seven different symbols represent Roman numerals with the following values:

| Symbol | Value |
|---|---|
| I | 1 |
| V | 5 |
| X | 10 |
| L | 50 |
| C | 100 |
| D | 500 |
| M | 1000 |

Roman numerals are formed by appending the conversions of decimal place values from highest to lowest. Converting a decimal place value into a Roman numeral has the following rules:

- If the value does not start with 4 or 9, select the symbol of the maximal value that can be subtracted from the input, append that symbol to the result, subtract its value, and convert the remainder to a Roman numeral.
- If the value starts with 4 or 9 use the **subtractive form** representing one symbol subtracted from the following symbol, for example, 4 is 1 (`I`) less than 5 (`V`): `IV` and 9 is 1 (`I`) less than 10 (`X`): `IX`. Only the following subtractive forms are used: 4 (`IV`), 9 (`IX`), 40 (`XL`), 90 (`XC`), 400 (`CD`) and 900 (`CM`).
- Only powers of 10 (`I`, `X`, `C`, `M`) can be appended consecutively at most 3 times to represent multiples of 10. You cannot append 5 (`V`), 50 (`L`), or 500 (`D`) multiple times. If you need to append a symbol 4 times use the **subtractive form**.

Given an integer, convert it to a Roman numeral.

**Example 1:**

    Input: num = 3749
    Output: "MMMDCCXLIX"
    Explanation:
    3000 = MMM as 1000 (M) + 1000 (M) + 1000 (M)
     700 = DCC as 500 (D) + 100 (C) + 100 (C)
      40 = XL as 10 (X) less of 50 (L)
       9 = IX as 1 (I) less of 10 (X)
    Note: 49 is not 1 (I) less of 50 (L) because the conversion is based on decimal places

**Example 2:**

    Input: num = 58
    Output: "LVIII"
    Explanation:
    50 = L
     8 = VIII

**Example 3:**

    Input: num = 1994
    Output: "MCMXCIV"
    Explanation:
    1000 = M
     900 = CM
      90 = XC
       4 = IV

**Constraints:**

- `1 <= num <= 3999`

---
## 🎯 Interview Walkthrough: Brute Force → Optimized

**Pattern:** Greedy with an ordered value table (include the 6 subtractive forms as 'coins')

| Approach | Time | Space |
|---|---|---|
| 1️⃣ Brute force (7 symbols + patch-up) | O(1)* | O(1) |
| 2️⃣ Optimized (13-entry greedy) | O(1)* | O(1) |

> 🗣️ lines are an **interview script**: roughly what to say out loud at each step.

### Step 0 · Clarify before coding

🗣️ *"I'm converting 1–3999 into a Roman string. The rules are: biggest symbols first, at most three repeats of I/X/C/M, and the six subtractive forms for 4s and 9s."*

- Range → `1 ≤ num ≤ 3999`. **This bounded range is why the complexity is O(1).**
- Uppercase output, no validation needed.
- Note from Example 1: `49` is `XLIX` (40 + 9), **not** `IL`. Subtraction happens **per decimal place**.

### Step 1 · Brute force

**Idea:** Greedily use only the **7 basic symbols** (like making change with coins 1000, 500, 100, …), which produces additive forms like `IIII` and `VIIII`. Then **patch** the forbidden patterns with string replacements (`DCCCC → CM`, `CCCC → CD`, …).

🗣️ *"A first idea: greedy with the seven basic symbols, then post-process the illegal runs into subtractive forms. It works, but the patch order matters (`VIIII` must become `IX` before `IIII` becomes `IV`), and that's a bug waiting to happen."*

In [1]:
class SolutionBrute:
    BASIC = [(1000, "M"), (500, "D"), (100, "C"), (50, "L"), (10, "X"), (5, "V"), (1, "I")]
    # order matters: 9-patterns before 4-patterns
    PATCHES = [("DCCCC", "CM"), ("CCCC", "CD"), ("LXXXX", "XC"), ("XXXX", "XL"), ("VIIII", "IX"), ("IIII", "IV")]

    def intToRoman(self, num: int) -> str:
        out = []
        for value, sym in self.BASIC:
            count, num = divmod(num, value)
            out.append(sym * count)
        s = "".join(out)
        for bad, good in self.PATCHES:
            s = s.replace(bad, good)
        return s

### Step 2 · Optimize

**Bottleneck:** The patch step is a fragile post-processing layer, and its order matters.

**Key insight: make the subtractive forms first-class "coins".** If the denomination list already contains `CM = 900`, `CD = 400`, `XC = 90`, `XL = 40`, `IX = 9`, `IV = 4`, then the greedy *never* produces `DCCCC` in the first place. With those 13 denominations, sorted descending, plain greedy is correct.

**Why greedy is correct here:** each decimal place is handled independently, and within a place the 13 values cover every digit 0–9 using at most 3 repeats. For example 8 = V+I+I+I, 9 = IX, 4 = IV. The greedy picks exactly the standard form for each digit.

🗣️ *"I'll put all 13 values, including the six subtractive pairs, into one list from largest to smallest. Then it's change-making: for each value, take as many copies as fit, append the symbol that many times, and subtract. No special cases and no patching. Because num is at most 3999, the output is at most 15 characters, so this is O(1)."*

#### Choosing the data structure & algorithm: why this, why not that

| Option | Verdict | Why / why not |
|---|---|---|
| 7 symbols + patch replacements | ⚠️ | Correct only with the right patch order. Easy to break. |
| **Ordered list of 13 (value, symbol) pairs + greedy** | ✅ | No special cases. The list order *is* the algorithm. |
| Per-digit lookup tables (`thousands[n//1000] + hundreds[n%1000//100] + …`) | ✅ (alternative) | Also O(1) and branch-free, but you hard-code 4 tables of 10 strings, which is easy to typo under pressure. |
| Dict instead of list | ❌ | We need a fixed **descending order**. A list of tuples states that explicitly. |
| Recursion | ❌ | No benefit over a simple loop. |

In [2]:
class Solution:
    VALUES = [
        (1000, "M"), (900, "CM"), (500, "D"), (400, "CD"),
        (100, "C"), (90, "XC"), (50, "L"), (40, "XL"),
        (10, "X"), (9, "IX"), (5, "V"), (4, "IV"), (1, "I"),
    ]

    def intToRoman(self, num: int) -> str:
        out = []
        for value, sym in self.VALUES:
            if num == 0:
                break
            count, num = divmod(num, value)   # how many of this "coin" fit
            out.append(sym * count)
        return "".join(out)

### Step 3 · Toy example walkthrough

`num = 1994`

| value | symbol | count = num // value | appended | num after |
|---|---|---|---|---|
| 1000 | M | 1 | M | 994 |
| 900 | CM | 1 | CM | 94 |
| 500 | D | 0 | n/a | 94 |
| 400 | CD | 0 | n/a | 94 |
| 100 | C | 0 | n/a | 94 |
| 90 | XC | 1 | XC | 4 |
| 50, 40, 10, 9, 5 | n/a | 0 | n/a | 4 |
| 4 | IV | 1 | IV | **0** → stop |

Result: `M` + `CM` + `XC` + `IV` = **`MCMXCIV`** ✅

`num = 3749` → MMM (3000) → D (500) → CC (200) → XL (40) → IX (9) = **`MMMDCCXLIX`** ✅

In [3]:
cases = [
    (3749, "MMMDCCXLIX"),
    (58, "LVIII"),
    (1994, "MCMXCIV"),
    (1, "I"),
    (4, "IV"),
    (9, "IX"),
    (49, "XLIX"),
    (3999, "MMMCMXCIX"),
]
for num, expected in cases:
    assert SolutionBrute().intToRoman(num) == expected
    assert Solution().intToRoman(num) == expected
    print(num, "->", expected)

# round-trip all values with a Roman -> int decoder
def from_roman(s):
    v = {"I": 1, "V": 5, "X": 10, "L": 50, "C": 100, "D": 500, "M": 1000}
    return sum(-v[c] if i + 1 < len(s) and v[c] < v[s[i + 1]] else v[c] for i, c in enumerate(s))

for n in range(1, 4000):
    r = Solution().intToRoman(n)
    assert from_roman(r) == n and r == SolutionBrute().intToRoman(n)
print("All tests passed ✅ (including all of 1..3999)")

3749 -> MMMDCCXLIX
58 -> LVIII
1994 -> MCMXCIV
1 -> I
4 -> IV
9 -> IX
49 -> XLIX
3999 -> MMMCMXCIX
All tests passed ✅ (including all of 1..3999)


### Step 4 · Wrap up

| | Time | Space | Why |
|---|---|---|---|
| 7 symbols + patch | O(1)* | O(1)* | Bounded output, but extra string passes |
| **13-entry greedy** | **O(1)*** | **O(1)*** | Fixed 13 iterations, output ≤ 15 chars |

\* **Complexity, explained:** the loop runs over a fixed list of 13 entries, and `num ≤ 3999` caps the output at 15 characters (`MMMDCCCLXXXVIII`). So the work is bounded by a constant, which is O(1). If the range were unbounded (allowing arbitrarily many `M`s), it would be O(num / 1000) for the leading `M`s.

**Edge cases:** 4s and 9s at every place (4, 9, 40, 90, 400, 900), 49 → `XLIX` (not `IL`), 3999.

**Follow-ups:**
- *Why does greedy work here but not for arbitrary coin systems?* → Roman values are "canonical" per decimal place. For coins like {1, 3, 4} and amount 6, greedy gives 4+1+1 but 3+3 is better. Be ready to say why this case is safe.
- *Reverse direction* → Roman to Integer (previous problem).

🗣️ **30-second recap:** *"I treat it like making change. I list all 13 values, including the six subtractive pairs like 900 = CM and 4 = IV, from largest to smallest. For each value, I append its symbol as many times as it fits and subtract. Including the subtractive forms means I never produce illegal runs like IIII. The input is bounded at 3999, so it's O(1) time and space."*